# 06 — Evaluation: Paper-Quality Measurement

세 모델 (Vanilla, Periodic, Trajectory)을 동일 protocol로 정밀 측정. Paper Section 4 (Experiments)에 들어갈 숫자/figure 산출.

**전 단계 대비 변경**:
- `n_seed = 10` (이전 3–5 → 통계적 유의성 확보)
- **Deterministic sampling** (`sampling_seed_base`로 RNG state 통제)
- 모든 raw 결과 → `eval_results.npz`로 저장 (분석/figure는 노트북 07에서)

**산출물**:
- **Table 1**: In-distribution performance (3 models × 1 freq × 10 seeds)
- **Table 2**: Controllability sweep (Trajectory only, 7 freqs × 10 seeds, Periodic 비교)
- **Figure 1**: Reward vs frequency (3-way comparison)

**Sub-contribution 3 (05)는 본 평가에서 제외** — future work으로 paper에서 별도 언급.

**예상 시간**: T4 기준 50분–1시간 30분 (총 100 rollout × 1000 step)

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import os, sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import (
    ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR,
    FIGURES_DIR, VIDEOS_DIR, ensure_artifact_dirs,
)
ensure_artifact_dirs()

print(f'✓ src 경로 등록: {SRC_DIR}')
print(f'✓ artifact root: {ARTIFACT_ROOT}')
print(f'  src 파일: {[f.name for f in SRC_DIR.glob("*.py")]}')


In [ ]:
!pip install -q -r {REPO_ROOT / 'requirements.txt'}
print("✓ requirements.txt 기반 의존성 준비 완료")

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel

from dataset  import load_project_data
from models   import (
    build_vanilla_dp_model,
    build_periodic_phase_dp_model,
    build_phase_trajectory_dp_model,
    count_params,
)
from training import load_checkpoint
from sampling import (
    rollout_multi_seed,
    vanilla_sample_cond_fn,
    periodic_phase_sample_cond_fn,
    trajectory_phase_sample_cond_fn,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__}, device={device}")

## 2. 데이터 로드

In [ ]:
data = load_project_data(DATA_DIR)
print(f"Freq window: {data['freq_window_mean']:.3f} ± {data['freq_window_std']:.3f} Hz")

SEED = data['seed']
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

## 3. 세 모델 동시 로드 + EMA

각각 독립 모델 객체. 모두 best_ema 적용. cond_fn은 별도로 매핑.

In [ ]:
# Noise scheduler config (모든 모델 공통)
NUM_TRAIN_TIMESTEPS = 100
NUM_INFERENCE_STEPS  = 16
ns_config = {
    'num_train_timesteps': NUM_TRAIN_TIMESTEPS,
    'beta_schedule':       'squaredcos_cap_v2',
    'prediction_type':     'epsilon',
    'clip_sample':         True,
}

# === Vanilla DP ===
m_vanilla = build_vanilla_dp_model(
    obs_dim=data['OBS_DIM'], act_dim=data['ACT_DIM'],
    obs_horizon=data['OBS_HORIZON'], device=device,
)
ema_vanilla = EMAModel(parameters=m_vanilla.parameters(), power=0.75)
load_checkpoint(CHECKPOINTS_DIR / 'vanilla_dp_ckpt.pt',
                m_vanilla, ema_vanilla, device=device, use_best_ema=True)

# === Periodic Phase DP ===
m_periodic = build_periodic_phase_dp_model(
    obs_dim=data['OBS_DIM'], act_dim=data['ACT_DIM'],
    obs_horizon=data['OBS_HORIZON'], device=device,
)
ema_periodic = EMAModel(parameters=m_periodic.parameters(), power=0.75)
load_checkpoint(CHECKPOINTS_DIR / 'phase_periodic_ckpt.pt',
                m_periodic, ema_periodic, device=device, use_best_ema=True)

# === Phase Trajectory DP ===
m_trajectory = build_phase_trajectory_dp_model(
    obs_dim=data['OBS_DIM'], act_dim=data['ACT_DIM'],
    obs_horizon=data['OBS_HORIZON'], per_step_cond_dim=2, device=device,
)
ema_trajectory = EMAModel(parameters=m_trajectory.parameters(), power=0.75)
load_checkpoint(CHECKPOINTS_DIR / 'phase_trajectory_ckpt.pt',
                m_trajectory, ema_trajectory, device=device, use_best_ema=True)

print(f"\n✓ 세 모델 로드 완료")
print(f"  Vanilla:    {count_params(m_vanilla)['trainable']/1e6:.2f}M")
print(f"  Periodic:   {count_params(m_periodic)['trainable']/1e6:.2f}M")
print(f"  Trajectory: {count_params(m_trajectory)['trainable']/1e6:.2f}M")

## 4. Phase trajectory function + env

In [ ]:
DT = 0.05
f_mean = data['freq_window_mean']

def make_phase_traj_fn(freq_hz: float, dt: float = DT):
    def phase_traj(current_step: int, pred_horizon: int):
        t_steps = np.arange(current_step, current_step + pred_horizon)
        phi = (2.0 * np.pi * freq_hz * t_steps * dt) % (2.0 * np.pi)
        return phi.astype(np.float32)
    return phase_traj

import gymnasium as gym
env = gym.make('Ant-v5')
print(f"Ant-v5 ready. f_mean={f_mean:.3f} Hz")

## 5. 평가 helper

세 모델 각각에 맞는 (model, ema, cond_fn, phase_kwargs) 튜플을 사전 정의하여 코드 중복 제거.

**중요**: `deterministic_sampling=True` — `sampling_seed_base = seed`로 매 rollout의 noise sampling이 deterministic. 같은 seed로 다시 돌리면 같은 결과 나옴.

In [ ]:
# 모델별 메타데이터
MODEL_SPECS = {
    'vanilla': {
        'model':   m_vanilla,
        'ema':     ema_vanilla,
        'cond_fn': vanilla_sample_cond_fn,
        'use_phase_traj': False,    # phase_trajectory_fn 안 사용
        'label':   'Vanilla DP',
    },
    'periodic': {
        'model':   m_periodic,
        'ema':     ema_periodic,
        'cond_fn': periodic_phase_sample_cond_fn,
        'use_phase_traj': True,
        'label':   'Periodic Phase',
    },
    'trajectory': {
        'model':   m_trajectory,
        'ema':     ema_trajectory,
        'cond_fn': trajectory_phase_sample_cond_fn,
        'use_phase_traj': True,
        'label':   'Trajectory (ours)',
    },
}

def eval_model(model_key: str, freq_hz: float, n_seeds: int = 10,
               max_steps: int = 1000) -> list:
    """Single model × single freq × n_seeds rollout.

    Returns list of result dicts (one per seed).
    Deterministic sampling enabled.
    """
    spec = MODEL_SPECS[model_key]
    phase_fn = make_phase_traj_fn(freq_hz) if spec['use_phase_traj'] else None

    return rollout_multi_seed(
        spec['model'], spec['ema'], env,
        n_seeds=n_seeds,
        deterministic_sampling=True,
        noise_scheduler_config=ns_config,
        obs_mean=data['obs_mean'], obs_std=data['obs_std'],
        act_min=data['act_min'], act_range=data['act_range'],
        cond_fn=spec['cond_fn'],
        obs_horizon=data['OBS_HORIZON'],
        pred_horizon=data['PRED_HORIZON'],
        action_horizon=data['ACTION_HORIZON'],
        obs_dim=data['OBS_DIM'],
        act_dim=data['ACT_DIM'],
        num_inference_steps=NUM_INFERENCE_STEPS,
        max_steps=max_steps,
        phase_trajectory_fn=phase_fn,
        device=device,
    )

print("eval_model() 정의 완료")

## 6. Table 1 — In-distribution performance

세 모델 × `f = 2.023 Hz` (학습 mean) × 10 seed × 1000 step.

**예상 결과 (이전 측정 기준)**:
| Model | Survival | Reward |
|---|---|---|
| Vanilla | ~1000 | ~330 |
| Periodic | ~1000 | ~670 |
| Trajectory | ~890 | ~1430 |

10 seed로 측정하면 std가 ~30% 줄어들어 차이 검정 가능.

In [ ]:
N_SEEDS_INDIST = 20

print(f"=== Table 1: In-dist @ f={f_mean:.3f} Hz, n={N_SEEDS_INDIST} seeds ===\n")
table1_results = {}
t0 = time.time()
for key in ['vanilla', 'periodic', 'trajectory']:
    print(f"\n--- {MODEL_SPECS[key]['label']} ---")
    table1_results[key] = eval_model(key, f_mean, n_seeds=N_SEEDS_INDIST, max_steps=1000)
print(f"\n총 시간: {(time.time()-t0)/60:.1f} min")

In [ ]:
# Table 1 정리
print(f"\n=== Table 1: In-distribution @ f={f_mean:.3f} Hz, n={N_SEEDS_INDIST} ===")
print(f"{'Model':>20s} | {'Survival':>15s} | {'Reward':>17s}")
print('-' * 60)
for key in ['vanilla', 'periodic', 'trajectory']:
    res = table1_results[key]
    surv = np.array([r['survival']     for r in res])
    rew  = np.array([r['total_reward'] for r in res])
    print(f"{MODEL_SPECS[key]['label']:>20s} | "
          f"{surv.mean():>5.0f} ± {surv.std():>4.0f}    | "
          f"{rew.mean():>7.1f} ± {rew.std():>5.1f}")

# Standard error (std/sqrt(n)) — 신뢰구간 정확도 진단
print(f"\n=== Standard Error (std/√n) ===")
for key in ['vanilla', 'periodic', 'trajectory']:
    res = table1_results[key]
    rew = np.array([r['total_reward'] for r in res])
    se = rew.std() / np.sqrt(len(rew))
    print(f"{MODEL_SPECS[key]['label']:>20s}: SE = {se:.1f}  ({se/rew.mean()*100:.1f}% of mean)")

## 7. Table 2 + Figure 1 — Controllability sweep

Periodic + Trajectory 두 모델 × 7 freq × 10 seed. Vanilla는 phase 받지 않으니 sweep 의미 없음 (제외).

**Periodic 결과는 paper의 weak baseline**, Trajectory는 main result.

**예상 (이전 결과 기준)**:
- Periodic: 모든 freq에서 평탄 (~700–900 reward, no trend)
- Trajectory: in-dist에서 inverted-U (1.92→2.13 peak ~1500), OOD-high에서 graceful degradation

In [ ]:
# 5 in-distribution interpolation points + 4 OOD stress points.
# 기존 mean±3σ sweep은 high OOD가 freq_window_max와 너무 가까워 구분력이 약했음.
f_min = float(data['freq_window_min'])
f_max = float(data['freq_window_max'])
f_width = f_max - f_min

in_freqs = np.linspace(f_min, f_max, 5, dtype=np.float32)
ood_freqs = (f_mean * np.array([0.50, 0.75, 1.25, 1.50], dtype=np.float32)).astype(np.float32)
ood_freqs = ood_freqs[ood_freqs > 0.2]  # physically meaningful lower bound

# 총 9개 freq를 명시적으로 사용: OOD-low 2개 + in-dist 5개 + OOD-high 2개.
sweep_freqs = np.concatenate([ood_freqs[:2], in_freqs, ood_freqs[2:]]).astype(np.float32)

assert len(in_freqs) == 5, f"Expected 5 interpolation freqs, got {len(in_freqs)}"
assert len(ood_freqs) == 4, f"Expected 4 OOD freqs, got {len(ood_freqs)}"
assert len(sweep_freqs) == 9, f"Expected 9 total sweep freqs, got {len(sweep_freqs)}"
assert np.all(ood_freqs[:2] < f_min) and np.all(ood_freqs[2:] > f_max), \
    f"OOD freqs must lie outside [{f_min:.3f}, {f_max:.3f}], got {ood_freqs}"

def freq_zone(freq_hz: float) -> str:
    if f_min <= freq_hz <= f_max:
        return 'in-dist'
    return 'OOD-low' if freq_hz < f_min else 'OOD-high'

N_SEEDS_SWEEP = 10
print(f"In-dist freqs: {in_freqs.round(3).tolist()}")
print(f"OOD freqs:     {ood_freqs.round(3).tolist()}")
print(f"Sweep freqs:   {sweep_freqs.round(3).tolist()}")
print(f"Zones:         {[freq_zone(float(f)) for f in sweep_freqs]}")
print(f"Models: periodic + trajectory, n={N_SEEDS_SWEEP} seeds each\n")

In [ ]:
# Periodic sweep
print(f"=== Periodic sweep ===")
sweep_periodic = {}
t0 = time.time()
for f in sweep_freqs:
    f = float(f)
    zone = freq_zone(f)
    print(f"\n--- freq={f:.3f} Hz ({zone}) ---")
    sweep_periodic[f] = eval_model('periodic', f,
                                   n_seeds=N_SEEDS_SWEEP, max_steps=1000)
print(f"\nPeriodic sweep 시간: {(time.time()-t0)/60:.1f} min")

In [ ]:
# Trajectory sweep
print(f"=== Trajectory sweep ===")
sweep_trajectory = {}
t0 = time.time()
for f in sweep_freqs:
    f = float(f)
    zone = freq_zone(f)
    print(f"\n--- freq={f:.3f} Hz ({zone}) ---")
    sweep_trajectory[f] = eval_model('trajectory', f,
                                     n_seeds=N_SEEDS_SWEEP, max_steps=1000)
print(f"\nTrajectory sweep 시간: {(time.time()-t0)/60:.1f} min")

## 8. Table 2 출력

In [ ]:
print(f"=== Table 2: Sweep @ n={N_SEEDS_SWEEP} ===")
print(f"{'freq':>6s} | {'zone':>8s} | "
      f"{'Periodic':>26s} | {'Trajectory':>26s} | Δreward")
print(f"{'':6s} | {'':8s} | "
      f"{'surv':>11s} {'reward':>13s} | "
      f"{'surv':>11s} {'reward':>13s}")
print('-' * 96)
for f in sweep_freqs:
    f = float(f)
    zone = freq_zone(f)
    p_res = sweep_periodic[f]
    t_res = sweep_trajectory[f]
    p_surv = np.array([r['survival']     for r in p_res])
    p_rew  = np.array([r['total_reward'] for r in p_res])
    t_surv = np.array([r['survival']     for r in t_res])
    t_rew  = np.array([r['total_reward'] for r in t_res])
    d_rew = t_rew.mean() - p_rew.mean()
    print(f"{f:>6.3f} | {zone:>8s} | "
          f"{p_surv.mean():>5.0f}±{p_surv.std():>3.0f}  "
          f"{p_rew.mean():>5.0f}±{p_rew.std():>4.0f} | "
          f"{t_surv.mean():>5.0f}±{t_surv.std():>3.0f}  "
          f"{t_rew.mean():>5.0f}±{t_rew.std():>4.0f} | "
          f"{d_rew:+7.0f}")

## 9. Figure 1 — Reward vs frequency (3-way comparison)

Paper의 main figure. Vanilla는 단일 점 (phase 안 받으니까), Periodic+Trajectory는 sweep line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

freqs_sorted = [float(f) for f in sweep_freqs]
freq_zones = [freq_zone(f) for f in freqs_sorted]

# Periodic
p_surv  = [np.mean([r['survival']     for r in sweep_periodic[f]]) for f in freqs_sorted]
p_surv_std = [np.std([r['survival']  for r in sweep_periodic[f]]) for f in freqs_sorted]
p_rew  = [np.mean([r['total_reward'] for r in sweep_periodic[f]]) for f in freqs_sorted]
p_rew_std  = [np.std([r['total_reward'] for r in sweep_periodic[f]]) for f in freqs_sorted]

# Trajectory
t_surv  = [np.mean([r['survival']     for r in sweep_trajectory[f]]) for f in freqs_sorted]
t_surv_std = [np.std([r['survival']  for r in sweep_trajectory[f]]) for f in freqs_sorted]
t_rew  = [np.mean([r['total_reward'] for r in sweep_trajectory[f]]) for f in freqs_sorted]
t_rew_std  = [np.std([r['total_reward'] for r in sweep_trajectory[f]]) for f in freqs_sorted]

# Vanilla — 단일 점 (in-dist mean에서만 측정됐으므로)
v_rew_indist  = np.array([r['total_reward'] for r in table1_results['vanilla']])
v_surv_indist = np.array([r['survival']     for r in table1_results['vanilla']])

# === Survival plot ===
ax = axes[0]
ax.errorbar(freqs_sorted, p_surv, yerr=p_surv_std,
            fmt='s-', capsize=4, linewidth=1.8, markersize=7,
            color='tab:green', label='Periodic Phase')
ax.errorbar(freqs_sorted, t_surv, yerr=t_surv_std,
            fmt='o-', capsize=4, linewidth=2.2, markersize=8,
            color='tab:red', label='Trajectory (ours)')
ax.errorbar([f_mean], [v_surv_indist.mean()], yerr=[v_surv_indist.std()],
            fmt='D', capsize=5, markersize=10, color='tab:blue',
            label='Vanilla (no phase, ref. only)')
ax.axvspan(f_min, f_max,
           alpha=0.12, color='green', label='In-dist range')
ax.axvline(f_mean, color='gray', ls='--', alpha=0.4)
ax.set_xlabel('Sampling-time phase freq (Hz)')
ax.set_ylabel('Survival (steps)')
ax.set_title(f'Survival vs Frequency (5 in-dist + 4 OOD, n={N_SEEDS_SWEEP})')
ax.legend(loc='lower center', fontsize=9); ax.grid(True, alpha=0.3)
ax.set_ylim(-50, 1080)

# === Reward plot ===
ax = axes[1]
ax.errorbar(freqs_sorted, p_rew, yerr=p_rew_std,
            fmt='s-', capsize=4, linewidth=1.8, markersize=7,
            color='tab:green', label='Periodic Phase')
ax.errorbar(freqs_sorted, t_rew, yerr=t_rew_std,
            fmt='o-', capsize=4, linewidth=2.2, markersize=8,
            color='tab:red', label='Trajectory (ours)')
ax.errorbar([f_mean], [v_rew_indist.mean()], yerr=[v_rew_indist.std()],
            fmt='D', capsize=5, markersize=10, color='tab:blue',
            label='Vanilla (no phase, ref. only)')
ax.axvspan(f_min, f_max,
           alpha=0.12, color='green', label='In-dist range')
ax.axvline(f_mean, color='gray', ls='--', alpha=0.4)
ax.set_xlabel('Sampling-time phase freq (Hz)')
ax.set_ylabel('Total reward')
ax.set_title(f'Reward vs Frequency (5 in-dist + 4 OOD, n={N_SEEDS_SWEEP})')
ax.legend(loc='best', fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
out_png = FIGURES_DIR / 'eval_figure1_reward_vs_freq.png'
plt.savefig(out_png, dpi=100, bbox_inches='tight')
plt.show()
print(f"✓ {out_png}")

## 10. 결과 저장 — `eval_results.npz`

분석/figure를 별도 노트북에서 다시 그릴 수 있도록 raw 결과 저장.

In [ ]:
# Helper: results list → 평탄한 numpy arrays
def stack_results(results_list):
    return {
        'survival':     np.array([r['survival']     for r in results_list]),
        'total_reward': np.array([r['total_reward'] for r in results_list]),
    }

freqs_sorted = [float(f) for f in sweep_freqs]
sweep_zone_labels = np.array([freq_zone(f) for f in freqs_sorted])

eval_results = {
    # Metadata
    'n_seeds_indist':   np.int32(N_SEEDS_INDIST),
    'n_seeds_sweep':    np.int32(N_SEEDS_SWEEP),
    'f_mean':           np.float32(f_mean),
    'freq_window_min':  np.float32(f_min),
    'freq_window_max':  np.float32(f_max),
    'in_freqs':         np.array(in_freqs, dtype=np.float32),
    'ood_freqs':        np.array(ood_freqs, dtype=np.float32),
    'sweep_freqs':      np.array(freqs_sorted, dtype=np.float32),
    'sweep_zone_labels': sweep_zone_labels,
}

# Table 1 (in-dist) — 3 models
for key in ['vanilla', 'periodic', 'trajectory']:
    s = stack_results(table1_results[key])
    eval_results[f'table1_{key}_survival'] = s['survival']
    eval_results[f'table1_{key}_reward']   = s['total_reward']

# Table 2 (sweep) — 2 models × 9 freqs (5 interpolation + 4 OOD)
sweep_p_surv = np.zeros((len(freqs_sorted), N_SEEDS_SWEEP), dtype=np.float32)
sweep_p_rew  = np.zeros((len(freqs_sorted), N_SEEDS_SWEEP), dtype=np.float32)
sweep_t_surv = np.zeros((len(freqs_sorted), N_SEEDS_SWEEP), dtype=np.float32)
sweep_t_rew  = np.zeros((len(freqs_sorted), N_SEEDS_SWEEP), dtype=np.float32)
for i, f in enumerate(freqs_sorted):
    s_p = stack_results(sweep_periodic[f])
    s_t = stack_results(sweep_trajectory[f])
    sweep_p_surv[i] = s_p['survival']
    sweep_p_rew[i]  = s_p['total_reward']
    sweep_t_surv[i] = s_t['survival']
    sweep_t_rew[i]  = s_t['total_reward']

eval_results['sweep_periodic_survival']   = sweep_p_surv
eval_results['sweep_periodic_reward']     = sweep_p_rew
eval_results['sweep_trajectory_survival'] = sweep_t_surv
eval_results['sweep_trajectory_reward']   = sweep_t_rew

out_path = RESULTS_DIR / 'eval_results.npz'
np.savez(out_path, **eval_results)
print(f"✓ Saved: {out_path} ({os.path.getsize(out_path)/1024:.1f} KB)")
print(f"\nKeys: {list(eval_results.keys())}")

## 11. 요약 + paper-quality 진단

**완료**:
- 세 모델 동일 protocol로 측정 (n=10, deterministic sampling)
- Table 1 (in-dist), Table 2 (sweep), Figure 1 (3-way comparison)
- Raw 결과 → `eval_results.npz`

**Paper 진단 — Standard Error 기준**:
- SE/mean < 15% → 차이 검정 가능, paper-quality
- SE/mean 15–25% → 추가 seed 필요할 수 있음
- SE/mean > 25% → noise dominates, n 더 늘리거나 narrative 변경

**다음 단계 (`07_paper_figures.ipynb`)**:
- `eval_results.npz` 로드 → paper figure 정밀 가공 (style, font, axis labels)
- LaTeX 표 자동 생성 (`tabulate` 등)
- Phase faithfulness figure (선택, 필요 시)